# GDL - Midterm n.1
**Student ID: 636526** (Your student "matricola" goes here)

**Submission ID: 004** (The ID from the sheet circulated in classroom goes here)

In the first midterm assignment, you are required to implement a Gaussian Mixture Model (GMM) and the Expectation-Maximization (EM) algorithm. You are allowed to use `numpy` and other non-machine learning libraries, e.g., `pandas`, `matplotlib`.

**Assumptions.** To ease the implementation, we assume, for each Gaussian distribution in the mixture $\mathcal{N}(\mu_k, \Sigma_k)$, that the *covariance matrix is diagonal*. Furthermore, keep in mind that a good implementation of the EM algorithm provides monotonically increasing log-likelihood, *but* can get stuck in local minima; initialization strategies are fundamental (random? sample points? k-means++?).

**Hyperparameter $k$.** When using a GMM, you should ask yourself: *how many mixtures will be in the data?* You can *maximize* the Bayesian Information Criterion (BIC) to select the number of categories $k$ on your training set. Formally,
$$
\text{BIC}
=
\log P(X\mid \theta) - \frac{|\theta|}{2}\log n,
$$
where $n$ is the number of samples in the training set, $\theta=\{\pi,\mu,\sigma\}$ the parameters of the GMM and $|\theta|$ the number of parameters, i.e., the sum of the number of parameters in $\pi$, $\mu$, and $\sigma$ (*hint*: it also depends on the dimensionality of data $d$!).

**Summary.** Overall, you are required to:
1.  **Implement the GMM class**. Fill the `log_likelihood(samples)`.
2. **Implement the EM algorithm**. Fill the `fit(samples)` method to fit the parameters of a GMM.
3. **Implement the BIC score**. Fill the `bic(samples)` method to perform model selection.
4. **Run training and evaluation**. Select the best scoring model using BIC (Bayesian Information Criterion) for values of $k=\{1,\ldots,6\}$.

**Evaluation.** Your solution will be tested against an hidden test set. For your learning experience, we require you to refrain from using LLM generated code. Violations will be flagged and invalidate the midterm. **Do not alter Sections 3 and 4 of the notebook.**

### 1. Libraries

In [ ]:
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)

print("Libraries imported successfully.")

### 2. Gaussian Mixture Model (GMM)

Feel free to play around the implementation. **For evaluation purposes**, the class **must** expose the following attributes and methods:
- `_pi: np.ndarray`
- `_mu: np.ndarray`
- `_sigma: np.ndarray`
- `fit(samples: np.ndarray)`
- `bic(samples: np.ndarray) -> np.ndarray`
- `log_likelihood(samples: np.ndarray) -> np.ndarray`

In [ ]:
class GaussianMixtureModel:
    def __init__(self, n_categories: int, n_features: int):
        self.n_categories = n_categories
        self.n_features = n_features    
        # Choose a smart initialization strategy for the parameters!
        # weights (pi), means (mu), covariances (sigma)
        # NOTE: for the covariance, you can store only the diagonal
        

    def log_likelihood(self, samples: np.ndarray) -> np.ndarray:
        """Computes logP(X|θ)"""
        total = 0.0
        # Loop over samples (Sum of logarithms)
        for j in range(samples.shape[0]):
            mixture = 0.0
            # Loop over categories (Weighted sum of Gaussians and pi)
            for k in range(self.n_categories):
                gaussian = 1.0
                # Loop over features (Product of multivariate Gaussians)
                for d in range(self.n_features):
                    gaussian *= 1/(np.sqrt(2 * np.pi * self._sigma[k, d])) * np.exp(-0.5 * (samples[j, d] - self._mu[k, d])**2 / self._sigma[k, d])
                mixture += self._pi[k] * gaussian   
            total += np.log(mixture)
        
        return total


    def fit(self, samples: np.ndarray):
        """Fits the GMM parameters using the EM algorithm."""
        
        # Initialize pi to be uniform
        self._pi = np.ones(self.n_categories) / self.n_categories
        print(f"\nShape of pi: {self._pi.shape}")
        print(f"Initial pi: {self._pi}") 

        # Initialize mu (random samples from data)
        indexes = np.random.choice(samples.shape[0], self.n_categories, replace=False)
        self._mu = samples[indexes]
        print(f"\nShape of mu: {self._mu.shape}")
        print(f"Initial mu: {self._mu}") 

        # Initialize sigma with data variance (only diagonal) 
        var = np.var(samples, axis=0)
        self._sigma = np.ones((self.n_categories, self.n_features)) * var
        print(f"\nShape of sigma: {self._sigma.shape}")
        print(f"Initial sigma: {self._sigma}")

        # EM algorithm
        iterations = 0
        ll_old = -np.inf

        # Break the loop after 100 iterations or if the LL converges 
        while (iterations < 100):
            # E-step: compute responsibilities
            responsibilities = np.zeros((samples.shape[0], self.n_categories))
            for j in range(samples.shape[0]):
                for k in range(self.n_categories):
                    responsibilities[j, k] = self._pi[k] * np.prod(1/np.sqrt(2 * np.pi * self._sigma[k]) * np.exp(-0.5 * (samples[j] - self._mu[k])**2 / self._sigma[k]))

            # Normalize responsibilities
            responsibilities /= responsibilities.sum(axis=1, keepdims=True)

            # M-step: update parameters
            Nk = responsibilities.sum(axis=0)
            self._pi = Nk / samples.shape[0]
            self._mu = np.dot(responsibilities.T, samples) / Nk[:, np.newaxis]
            
            for k in range(self.n_categories):
                diff = samples - self._mu[k]
                self._sigma[k] = np.dot(responsibilities[:, k], diff**2) / Nk[k]

            # Log-likelihood compute and convergence check
            ll_new = self.log_likelihood(samples)
            print(f"Iteration {iterations}, Log-Likelihood: {ll_new}")
            if np.abs(ll_new - ll_old) < 0.0001:
                break
            ll_old = ll_new
            iterations += 1


    def bic(self, samples: np.ndarray) -> np.ndarray:
        """Computes the BIC score."""
        
        ll = self.log_likelihood(samples)
        c = self.n_categories
        f = self.n_features
        # Pi = c - 1 , Mu = c * f, Sigma = c * f
        n_params = c - 1 + 2 * c * f
        score = ll - (n_params / 2) * np.log(samples.shape[0])
        return score


### 3. Training

Trains the model for increasing number of categories and selects the best scoring one in terms of BIC score.

In [ ]:
# load training set
train_df = pd.read_csv('train.csv')
print(f"train.csv loaded successfully. n={train_df.shape[0]}, d={train_df.shape[1]}")

# model selection using bayesian information score
seed_everything(9951)
candidate_categories = range(1, 7)
max_bic, best_k, best_gmm = -np.inf, None, None
for k in candidate_categories:
    gmm = GaussianMixtureModel(n_categories=k, n_features=train_df.shape[1])
    gmm.fit(train_df.values)
    ll = gmm.log_likelihood(train_df.values)
    bic_score = gmm.bic(train_df.values)
    print(f"k={k}\tBIC={bic_score:.4f}\tlogP(X|θ)={ll:.4f}")
    if bic_score > max_bic:
        max_bic = bic_score
        best_k = k
        best_gmm = gmm

# print training info
print()
print("Best model")
print(f"k:\t\t{best_k}")
print(f"BIC:\t\t{max_bic:.4f}")
print(f"logP(X|θ):\t{best_gmm.log_likelihood(train_df.values):.4f}")

### 4. Evaluation

To check if you did not break the API, you can use the training file to run the tests.

This is not meaningful for the evaluation of your model, as the true test set is hidden.

In [ ]:
# If test.csv does not exist copy train into test
!test -f test.csv || cp train.csv test.csv

In [ ]:
# load hidden test set ☠️
test_df = pd.read_csv('test.csv')
print(f"test.csv loaded successfully. n={test_df.shape[0]}, d={test_df.shape[1]}")

# print log-likelihood
test_log_likelihood = best_gmm.log_likelihood(test_df.values)
print(f"Log-likelihood of test data: {test_log_likelihood:.4f}")

# print parameters
print(f"k: {best_gmm.n_categories}")
print()
for cat_id in range(best_gmm.n_categories):
  print(f"π[{cat_id}]:", best_gmm._pi[cat_id])
  print(f"μ[{cat_id}]:", best_gmm._mu[cat_id])
  print(f"σ[{cat_id}]:", best_gmm._sigma[cat_id])
  print()